# Finding Optimal Routes

Now we apply Dijkstra's to a real problem: **finding the fastest shipping routes**.

**The business question:**
Our freight forwarding company has 5 months of shipping data. We know which routes we've historically used—but are they optimal? 

Could we be faster?

**What we'll do:**
1. Project the logistics network with aggregated transit times
2. Find the optimal route between specific origin-destination pairs
3. Query multiple destinations efficiently
4. Identify the gap between "what Dijkstra recommends" and "what we've been doing"

Let's start by connecting to our analytics environment.

## Imports

In [ ]:
import os
import pandas as pd
from datetime import timedelta
from IPython.display import display
from dotenv import load_dotenv
from graphdatascience.session import GdsSessions, AuraAPICredentials, DbmsConnectionInfo, SessionMemory

## Setup: Connect to AGA

In [ ]:
# Load environment variables
load_dotenv()

# Get Aura API credentials
client_id = os.getenv('AURA_CLIENT_ID')
client_secret = os.getenv('AURA_CLIENT_SECRET')
project_id = os.getenv('AURA_PROJECT_ID')  # set in .env only if your Aura account has multiple projects

# Get AuraDB connection info
uri = os.getenv('AURA_URI')
username = os.getenv('AURA_USERNAME')
password = os.getenv('AURA_PASSWORD')

In [ ]:
# Create sessions manager
sessions = GdsSessions(
    api_credentials=AuraAPICredentials(client_id, client_secret, project_id=project_id)
)

print("Sessions manager created")

In [ ]:
# Create a GDS Session
gds = sessions.get_or_create(
    session_name="logistics-routing",
    memory=SessionMemory.m_2GB,
    db_connection=DbmsConnectionInfo(
        uri=uri,
        username=username,
        password=password
    ),
    ttl=timedelta(minutes=30)
)

gds.verify_connectivity()
print(f"Connected to GDS Session: logistics-routing")

In [ ]:
from workshop_helpers import configure, visualize_query, visualize_projection

configure(gds, uri, username, password, os.getenv("AURA_DATABASE") or "neo4j")
print("Helper functions loaded: visualize_query(), visualize_projection()")

## Dataset recap

We'll work with the Cargo 2000 freight forwarding dataset—real air cargo logistics data from IATA.

This represents tracking and tracing events from a forwarding company over five months: business processes where smaller shipments are consolidated and shipped together to customers.

* `EntryPoint` — Source airports where freight originates
* `DepartureWarehouse` / `ArrivalWarehouse` — Processing facilities
* `TransferPoint` — Connection points between legs
* `Destination` — Final delivery airports

And relationship types:

* `RECEPTION`, `DEPARTURE`, `TRANSPORT`, `DELIVERY`

Each relationship has an `effectiveMinutes` property—actual time for that step.

In [ ]:
# Example: Three incoming legs converge at a transfer point, then deliver to destination
VG = visualize_query("""
MATCH path = (origin:DeparturePoint)-[:RECEPTION|DEPARTURE|TRANSPORT*1..3]->(transfer)-[:DELIVERY]->(dest:Destination {name: 'Davisfort'})
WHERE origin.name IN ['Davidburgh', 'Bryanside', 'Moodytown']
RETURN path
LIMIT 10
""")
VG.render()

## Step 1: Project the Graph

We'll project the **raw operations layer** of our data—the actual logistics steps.

**Why filter out HistoricalRoute nodes?**
The dataset has two layers:
- Raw operations: EntryPoint → warehouses → TransferPoints → Destination
- Historical summaries: HistoricalRoute nodes that record which routes we've used

For pathfinding, we want the raw operations—Dijkstra's will find optimal paths 
through the actual network topology. We'll compare against HistoricalRoutes later.

The raw graph has ~58,700 relationship instances—many representing the same 
route taken multiple times with different durations.

**What we do:** Aggregate parallel relationships into one, with averaged transit time.
- Raw: `effectiveMinutes` on each individual shipment
- Aggregated: `avgMinutes` = average across all shipments on that route

**Why average?**
Individual shipments vary due to weather, customs, operational issues. The 
average represents typical performance—a reasonable baseline for recommendations.

**Result:** ~2,024 relationships instead of ~58,700. Much more efficient for 
pathfinding, and Dijkstra's won't double-count routes we've used multiple times.

In [ ]:
# Project the logistics network with aggregated transit times
G, result = gds.graph.project(
    "logistics-network",
    """
    CALL {
        MATCH (source)
        WHERE source:EntryPoint OR source:DeparturePoint OR source:DepartureWarehouse
           OR source:TransferPoint OR source:ArrivalWarehouse OR source:Destination
        OPTIONAL MATCH (source)-[r:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY]->(target)
        WITH source, target, type(r) AS relType,
             avg(r.effectiveMinutes) AS avgMinutes  // Aggregate transit times
        RETURN source, target, relType, avgMinutes
    }
    RETURN gds.graph.project.remote(source, target, {
        sourceNodeLabels: labels(source),
        targetNodeLabels: labels(target),
        relationshipType: relType,
        relationshipProperties: {avgMinutes: avgMinutes}
    })
    """
)

print(f"Projected graph: {G.name()}")
print(f"  Nodes: {G.node_count():,}")
print(f"  Relationships: {G.relationship_count():,}")

### Visualize: Sample of the Logistics Network

Now let's take a look at our projection:

In [ ]:
# Visualize a sample of the network
VG = visualize_projection(G)
VG.render()

## Step 2: Run Dijkstra's

Find the optimal route from Howardborough to Ramoshaven.

First, we'll find our source and target nodeIds using `gds.find_node_id`

In [ ]:
# Find source and target node IDs
source_id = gds.find_node_id(["EntryPoint"], {"name": "Howardborough"})
target_id = gds.find_node_id(["Destination"], {"name": "Ramoshaven"})

print(f"Source: Howardborough (ID: {source_id})")
print(f"Target: Ramoshaven (ID: {target_id})")

Next, we run Dijskra's, using our nodeIds as the source and target.

In [ ]:
# Run Dijkstra's algorithm
dijkstra_result = gds.shortestPath.dijkstra.stream(
    G,
    sourceNode=source_id,
    targetNode=target_id,
    relationshipWeightProperty='avgMinutes'
)

# Get path details
path_node_ids = dijkstra_result['nodeIds'].iloc[0]
total_cost = dijkstra_result['totalCost'].iloc[0]

# Get node names along the path
path_details = gds.run_cypher("""
    UNWIND $nodeIds AS nodeId
    RETURN gds.util.asNode(nodeId).name AS name
""", params={"nodeIds": list(path_node_ids)})

# Remove duplicate consecutive names
route = []
prev_name = None
for name in path_details['name'].tolist():
    if name != prev_name:
        route.append(name)
        prev_name = name

print(f"Optimal Route: {' -> '.join(route)}")
print(f"\nTotal transit time:")
print(f"  {round(total_cost):,} minutes")
print(f"  {round(total_cost / 60, 2)} hours")
print(f"  {round(total_cost / 1440, 2)} days")

Let's take a look at that:

In [ ]:
# Get location names from the path
names_df = gds.run_cypher("""
    UNWIND $nodeIds AS nodeId
    RETURN gds.util.asNode(nodeId).name AS name
""", params={"nodeIds": list(path_node_ids)})
names = list(set(names_df['name'].tolist()))

# Visualize the optimal route
print(f"Optimal Route: Howardborough → Ramoshaven ({round(total_cost)} min)")
VG = visualize_query(f"""
    MATCH path = SHORTEST 1 (a)-[r:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY]->(b)
    WHERE a.name IN {names} AND b.name IN {names}
    RETURN path
""")
VG.render()

## Interpreting the Results

Dijkstra's found the most efficient path **through intermediate airports**.

* **route**: The sequence of locations from origin to destination
* **total_minutes**: Total transit time across all legs
* **total_hours/days**: Human-readable duration

This is the theoretically optimal journey based on historical averages.

Dijkstra's finds whichever path minimizes total time—whether that's direct or through multiple transfer points.

So, if we were looking for just one new route from pointA to pointB, we could find it this way. But what if we want to find multiple shortest paths to a variaty of destinations? 

In [ ]:
# Check if we have historical data for this route
historical_check = gds.run_cypher("""
    MATCH (e:EntryPoint {name: 'Howardborough'})-[:HAS_HISTORICAL_ROUTE]->(hr:HistoricalRoute)-[:TERMINATES_AT]->(d:Destination {name: 'Ramoshaven'})
    RETURN count(hr) AS historical_routes_used
""")

print(f"Historical routes we've used for Howardborough → Ramoshaven: {historical_check['historical_routes_used'].iloc[0]}")
print(f"Dijkstra's optimal route: {round(total_cost)} minutes")
print("\nTo properly compare: we need to see MULTIPLE top routes and check if our")
print("historical choices are among them. That's what Yen's algorithm does—next lesson!")

## Step 3: Multiple Destinations

Find optimal routes from Howardborough to several destinations.

None of these destinations have direct flights from Howardborough—Dijkstra's finds the best path through the network.

In [ ]:
# Find source and multiple destination node IDs
destinations = ['Ramoshaven', 'Jasmineside', 'Seanfurt', 'Meganbury', 'Hullport']

source_id = gds.find_node_id(["EntryPoint"], {"name": "Howardborough"})
target_ids = [gds.find_node_id(["Destination"], {"name": dest}) for dest in destinations]

print(f"Source: Howardborough")
print(f"Targets: {', '.join(destinations)}")

In [ ]:
# Run Dijkstra's for multiple destinations
multi_result = gds.shortestPath.dijkstra.stream(
    G,
    sourceNode=source_id,
    targetNodes=target_ids,
    relationshipWeightProperty='avgMinutes'
)

# Display results
results = []
for idx, row in multi_result.iterrows():
    # Get route names
    path_details = gds.run_cypher("""
        UNWIND $nodeIds AS nodeId
        RETURN gds.util.asNode(nodeId).name AS name
    """, params={"nodeIds": list(row['nodeIds'])})
    
    # Remove duplicate consecutive names
    route = []
    prev_name = None
    for name in path_details['name'].tolist():
        if name != prev_name:
            route.append(name)
            prev_name = name
    
    results.append({
        'destination': route[-1],
        'route': ' → '.join(route),
        'total_minutes': round(row['totalCost']),
        'total_hours': round(row['totalCost'] / 60, 2)
    })

# Sort by total_minutes
results_df = pd.DataFrame(results).sort_values('total_minutes')
display(results_df)

### Visualize: All Routes from Howardborough

In [ ]:
# Collect all location names from all paths
all_node_ids = set()
for idx, row in multi_result.iterrows():
    all_node_ids.update(row['nodeIds'])

# Get location names
names_df = gds.run_cypher("""
    UNWIND $nodeIds AS nodeId
    RETURN gds.util.asNode(nodeId).name AS name
""", params={"nodeIds": list(all_node_ids)})
names = list(set(names_df['name'].tolist()))

# Visualize all routes
print("All Optimal Routes from Howardborough")
VG = visualize_query(f"""
    MATCH path = SHORTEST 1 (a)-[r:RECEPTION|DEPARTURE|TRANSPORT|DELIVERY]->(b)
    WHERE a.name IN {names} AND b.name IN {names}
    RETURN path
""")
VG.render()

Now, we have a breakdown of the most efficient routes to various destinations from pointA. 

## The Limitation

Dijkstra's finds **one** optimal route per destination—great for something like single-journey planning.

But real logistics questions are more complex:

* What are the **alternative** routes?
* How do our **historical choices** compare?
* Are we missing faster or cheaper options we've never tried?

To answer these, we need to see **multiple** top routes—not just the best one.

## What We Can't Answer Yet

For any origin-destination pair, there may be many viable routes through different transfer points.

Some questions we can't yet answer:

* What's the **second-best** route if the optimal one is at capacity?
* Which historically-used routes are **not** in the top recommendations?
* How much time could we save by **switching** routes?

Dijkstra's only returns the single best path—we need more.

## Cleanup

In [ ]:
# Drop all projections
for graph_name in gds.graph.list()["graphName"].tolist():
    gds.graph.drop(graph_name)
    print(f"Dropped: {graph_name}")

In [ ]:
# Delete the session
gds.delete()
print("Session deleted - billing stopped")

## Summary

You've applied Dijkstra's to real logistics data:

* Projected a network with aggregated transit times
* Found optimal routes through intermediate transfer points  
* Queried multiple destinations efficiently
* Understood why aggregation matters for performance

**What Dijkstra's answered:** "What's the single fastest route?"

**What we still need to answer:**
- What's the second-best route if the optimal one is at capacity?
- What are ALL the competitive routes for a given origin-destination?
- Are our historically-used routes even in the top 5? Top 10?
- How much time could we save by switching?

Dijkstra's gives us **one** answer. Real logistics operations need options.


In the next lesson, you'll use **Yen's algorithm** to find multiple ranked routes—and discover that some historically popular routes aren't even in the top 10.